In [3]:
import pandas as pd
import numpy as np

class DecisionID3Tree:
    def __init__(self):
        self.tree = None

    def entropy(self, data_df, target):
        classes = data_df[target].unique()
        info_sum = 0
        total = len(data_df)
        for one_class in classes:
            cl_count = len(data_df[data_df[target] == one_class])
            p = cl_count / total
            if p != 0:
                info_sum += -p * np.log2(p)
        return info_sum

    def attribute_gain(self, attribute, dataset, target):
        cats = dataset[attribute].unique()
        info_att = 0
        for cat in cats:
            subset = dataset[dataset[attribute] == cat]
            ent = self.entropy(subset, target)
            weight = len(subset) / len(dataset)
            info_att += weight * ent
        info_gained = self.entropy(dataset, target) - info_att
        return info_gained

    def binary_threshold(self, data, target, attribute):
        values = np.sort(data[attribute].unique())
        mid_points = []
        for index in range(len(values)):
            if index != 0:
                mid_points.append((values[index] + values[index - 1]) / 2)
        best_info_gained = -1
        threshold = None
        aux_df = pd.DataFrame()
        aux_df[target] = data[target].values
        for mid_point in mid_points:
            aux_df["aux"] = np.where(data[attribute] > mid_point, f'>{mid_point}', f'<={mid_point}')
            aux_gained = self.attribute_gain("aux", aux_df, target)
            if aux_gained > best_info_gained:
                threshold = mid_point
                best_info_gained = aux_gained
        return threshold

    def best_att(self, dataset, target, attrs):
        dataset = dataset.copy()
        gain_df = pd.DataFrame(columns=["attribute", "info_gained"])
        num_attrs = []
        for attr in attrs:
            working_attr = attr
            if dataset[attr].dtype.kind in "iuf":
                thres = self.binary_threshold(dataset, target, attr)
                if thres is not None:
                    dataset[f'binary_{attr}'] = np.where(dataset[attr] > thres, f'>{thres}', f'<={thres}')
                    working_attr = f'binary_{attr}'
                    num_attrs.append(working_attr)
            attr_gain = self.attribute_gain(working_attr, dataset, target)
            row = {"attribute": working_attr, "info_gained": attr_gain}
            gain_df = pd.concat([gain_df, pd.DataFrame([row])], ignore_index=True)
        best_attribute = gain_df.loc[gain_df["info_gained"].idxmax(), "attribute"]
        for num_attr in num_attrs:
            if best_attribute == num_attr:
                raw_att = best_attribute.replace('binary_', '')
                dataset = dataset.drop(columns=[raw_att])
            else:
                dataset = dataset.drop(columns=[num_attr])
        return dataset, best_attribute

    def most_frequent_class(self, data, tar):
        classes = data[tar].unique()
        best_count = -1
        best_class = None
        for a_class in classes:
            class_count = len(data[data[tar] == a_class])
            if class_count > best_count:
                best_class = a_class
                best_count = class_count
        return best_class

    def build(self, dataset, target, attributes):
        if len(dataset[target].unique()) == 1:
            return dataset[target].unique()[0]
        if len(attributes) == 0:
            return self.most_frequent_class(dataset, target)
        df2, best_attribute = self.best_att(dataset, target, attributes)
        tree = {best_attribute: {}}
        cats = df2[best_attribute].unique()
        raw_best = best_attribute.replace('binary_', '')
        new_attributes = [att for att in attributes if att != raw_best]
        for cat in cats:
            subset = df2[df2[best_attribute] == cat]
            if len(subset) == 0:
                tree[best_attribute][cat] = self.most_frequent_class(df2, target)
            else:
                tree[best_attribute][cat] = self.build(subset, target, new_attributes)
        return tree

    def fit(self, dataset, target):
        attributes = [col for col in dataset.columns if col != target]
        self.tree = self.build(dataset, target, attributes)
        return self.tree

    def show(self, tree=None, indent=""):
        if tree is None:
            tree = self.tree
        if not isinstance(tree, dict):
            print(f"{indent}  -> Result: {tree}")
            return
        for attr, branches in tree.items():
            print(f"{indent}[{attr}]")
            for val, subtree in branches.items():
                print(f"{indent}  val: {val}")
                self.show(subtree, indent + "    ")

In [4]:
my_data = {
    "Refund": ["Yes", "No", "No", "No", "No", "No", "Yes", "No", "No", "No"],
    "Marital_Status": ["Single", "Married", "Single", "Married", "Divorced", "Married", "Divorced", "Single", "Married", "Single"],
    "Taxable_Income": [125000, 100000, 70000, 120000, 95000, 60000, 220000, 85000, 75000, 90000],
    "Cheat": ["No", "No", "No", "No", "Yes", "No", "No", "Yes", "No", "Yes"]}
my_data=pd.DataFrame(my_data)
my_target="Cheat"

my_model=DecisionID3Tree()
my_model.fit(my_data,my_target)
my_model.show()

[binary_Taxable_Income]
  val: >97500.0
      -> Result: No
  val: <=97500.0
    [Marital_Status]
      val: Single
        [Refund]
          val: No
              -> Result: Yes
      val: Divorced
          -> Result: Yes
      val: Married
          -> Result: No


In [7]:
my_data.sort_values(by='Taxable_Income')

,Refund,Marital_Status,Taxable_Income,Cheat
5,No,Married,60000,No
2,No,Single,70000,No
8,No,Married,75000,No
7,No,Single,85000,Yes
9,No,Single,90000,Yes
4,No,Divorced,95000,Yes
1,No,Married,100000,No
3,No,Married,120000,No
0,Yes,Single,125000,No
6,Yes,Divorced,220000,No
